# WaveLab RL trên Colab
Nếu cell kiểm tra Python không in thông báo, hãy kiểm tra nút Connect/Kết nối của Colab và runtime GPU. Chạy từng cell theo thứ tự; không Run all vì cell STOP ở cuối sẽ yêu cầu dừng train.


## Kế hoạch hiện tại — 14/11/2026
1. Chạy `parameter_progress` 4k bước trên 256 mẫu, đánh giá trên 32 mẫu riêng.
2. Chờ train và đánh giá xong; đổi `REWARD_MODE` sang `absolute`, chạy cùng ngân sách trong thư mục riêng.
3. So sánh MARE từng tham số, sai số tổng hợp và nghiệm cuối với nghiệm best-J. Đây là pilot một seed; chưa train dài trước khi có kết quả.
Giữ batch8, 4 môi trường, encoder epoch50 đóng băng và khởi tạo ngẫu nhiên để gần phép thử laptop. Nếu đổi batch/workers phải xem là cấu hình khác.


In [ ]:
print('Python đang hoạt động. Tiếp theo sẽ yêu cầu kết nối Google Drive.', flush=True)
from google.colab import drive
drive.mount('/content/drive')
print('Đã kết nối Google Drive.', flush=True)

In [ ]:
import subprocess,sys,os,json,hashlib,shutil,time
from pathlib import Path
REPO=Path('/content/wavelab-rl-scm')
def run_visible(command):
    print('Đang chạy:', ' '.join(map(str,command)), flush=True)
    child=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in child.stdout: print(line,end='',flush=True)
        status=child.wait()
        if status: raise RuntimeError(f'Lệnh thất bại, exit code {status}')
    except BaseException:
        child.terminate()
        raise
if not REPO.exists(): run_visible(['git','clone','--depth','1','https://github.com/hongphuc-git/wavelab-rl-scm.git',str(REPO)])
else: print('Repo đã có:',REPO,flush=True)
print('Hoàn tất bước tải code.', flush=True)

In [ ]:
print('Kiểm tra GPU và PyTorch có sẵn...',flush=True)
import torch,torchvision
print('Torch:',torch.__version__,'Torchvision:',torchvision.__version__,flush=True)
assert torch.cuda.is_available(), 'Chọn Runtime → Change runtime type → GPU rồi kết nối lại'
print('GPU:',torch.cuda.get_device_name(0),flush=True)
print('Cài thư viện RL, log sẽ hiển thị bên dưới...',flush=True)
run_visible([sys.executable,'-m','pip','install','--disable-pip-version-check','--timeout','60','--retries','2','-r',str(REPO/'requirements-colab.txt')])
print('Cài đặt xong. Có thể chạy cell Cấu hình.',flush=True)

## Cấu hình
Epoch 100 mặc định. Đổi thành 50 để đối chứng. Giữ torch/torchvision có sẵn nếu import được.

In [ ]:
EPOCH=50
REWARD_MODE="parameter_progress"  # chạy xong và đánh giá rồi đổi thành absolute
TOTAL_STEPS=4000
DRIVE=Path('/content/drive/MyDrive')
SOURCE=DRIVE/'conv/Bone_Phantom_Training_Data_b8.h5'
WEIGHTS=DRIVE/f'conv/18sep/pipe_efficientnet_v2_l/epoch_{EPOCH}_checkpoint/pytorch_model.bin'
DRIVE_RUN=DRIVE/f'WaveLab_RL/efficientnet_v2_l_epoch{EPOCH}/reward_compare256_{REWARD_MODE}_seed2026'
DRIVE_RUN.mkdir(parents=True,exist_ok=True)
RUN=Path(f'/content/rl_runs/epoch{EPOCH}_reward256_{REWARD_MODE}')
RUN.mkdir(parents=True,exist_ok=True)
assert SOURCE.exists() and WEIGHTS.exists(), 'Kiểm tra đường dẫn Drive'
LOCAL_WEIGHTS=Path(f'/content/epoch{EPOCH}.bin')
if not LOCAL_WEIGHTS.exists(): shutil.copyfile(WEIGHTS,LOCAL_WEIGHTS)
os.environ['WAVEDL_ENCODER_CHECKPOINT']=str(LOCAL_WEIGHTS)
os.environ['RLSCM_BACKEND']='python'
for name in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS'): os.environ[name]='1'
STOP=RUN/'STOP'
os.environ['RLSCM_STOP_FILE']=str(STOP)
# Restore only on a new local runtime; preserve matching replay metadata.
if not any(RUN.glob('checkpoint_*_steps.zip')):
    candidates=sorted(DRIVE_RUN.glob('checkpoint_*_steps.zip'),key=lambda p:int(p.stem.split('_')[1]))
    if candidates:
        newest=candidates[-1]
        shutil.copyfile(newest,RUN/newest.name)
        meta=DRIVE_RUN/'replay_buffer.json'
        if meta.exists() and json.loads(meta.read_text()).get('step')==int(newest.stem.split('_')[1]):
            for name in ('replay_buffer.pkl','replay_buffer.json'):shutil.copyfile(DRIVE_RUN/name,RUN/name)
print('Train trên ổ cục bộ:',RUN,'; sao lưu:',DRIVE_RUN)

In [ ]:
import numpy as np,h5py
DATA=RUN/'train256.h5'
VALID=RUN/'validation32.h5'
if not DATA.exists():
    with h5py.File(SOURCE,'r') as f:
        counts=np.asarray(f['nnz_per_matrix']).ravel()
        ids=np.random.default_rng(20261113).choice(np.flatnonzero(counts>0),288,replace=False)
        labels=np.asarray(f['samples'])[:,ids].T
        H=int(np.asarray(f.attrs['grid_H']).item()); W=int(np.asarray(f.attrs['grid_W']).item())
        images=np.zeros((288,256,256),np.uint8)
        for j,i in enumerate(ids):
            n=int(counts[i]);r=f['row_indices'][i,:n].astype(np.int64);c=f['col_indices'][i,:n].astype(np.int64)
            images[j,np.minimum(r*256//H,255),np.minimum(c*256//W,255)]=1
        with h5py.File(DATA,'w') as g:
            g.attrs['split']='train';g.create_dataset('images',data=images[:256]);g.create_dataset('parameters',data=labels[:256]);g.create_dataset('complete',data=np.ones(256,bool));g.create_dataset('source_indices',data=ids[:256])
        with h5py.File(VALID,'w') as g:
            g.attrs['split']='validation';g.create_dataset('images',data=images[256:]);g.create_dataset('parameters',data=labels[256:]);g.create_dataset('complete',data=np.ones(32,bool));g.create_dataset('source_indices',data=ids[256:])
    manifest=dict(source=str(SOURCE),source_indices=ids.tolist(),geometry='free pipe b=8 mm',preprocessing='Original WaveDL floor mapping; binary 0/1 256x256',checkpoint=str(WEIGHTS),checkpoint_sha256=hashlib.sha256(LOCAL_WEIGHTS.read_bytes()).hexdigest(),diagnostic='256 train / 32 disjoint validation; one seed pilot; no independent test claim')
    (RUN/'manifest.json').write_text(json.dumps(manifest,indent=2))
print('Dataset:',DATA)

## Bắt đầu hoặc resume
Train chạy nền. Không chạy cell hai lần khi tiến trình còn sống. Mỗi 1k bước lưu model và replay vào Drive. Colab có thể mất kết nối; dùng cell này để resume.

In [ ]:
if 'process' in globals() and process.poll() is None: raise RuntimeError('Đang train')
assert SOURCE.exists() and WEIGHTS.exists(), 'Drive không truy cập được; kết nối lại Drive trước'
RUN.mkdir(parents=True,exist_ok=True)
probe=RUN/'write_probe.tmp'
probe.write_text('ok');probe.unlink()
print('Drive ghi được, bắt đầu train...',flush=True)
checkpoints=sorted(RUN.glob('checkpoint_*_steps.zip'),key=lambda p:int(p.stem.split('_')[1]))
latest=checkpoints[-1] if checkpoints else None
done=int(latest.stem.split('_')[1]) if latest else 0
assert done<TOTAL_STEPS, 'Đã đạt ngân sách'
STOP.unlink(missing_ok=True)
initial_lr=1e-4;final_lr=9.76e-5
resume_lr=initial_lr+(final_lr-initial_lr)*done/TOTAL_STEPS
cmd=[sys.executable,'-u',str(REPO/'scripts/train_bone_sac_parallel.py'),'--dataset',str(DATA),'--steps',str(TOTAL_STEPS-done),'--n-envs','4','--encoder',f'wavedl_epoch{EPOCH}','--share-encoder','--param-weight','2','--reward-mode',REWARD_MODE,'--horizon','8','--action-scale','0.15','--buffer-size','2000','--batch-size','8','--learning-starts','128','--learning-rate',str(resume_lr),'--lr-final',str(final_lr),'--save-every','1000','--save-replay','--seed','2026','--device','cuda','--torch-threads','2','--out',str(RUN)]
if latest:
    saved=json.loads((RUN/'run_config.json').read_text())
    assert saved.get('reward_mode','absolute')==REWARD_MODE, 'Không resume khác reward'
    cmd+=['--resume',str(latest)]
else: cmd+=['--entropy-coef','0.01','--save-initial']
session_log_offset=(RUN/'training.log').stat().st_size if (RUN/'training.log').exists() else 0
log=(RUN/'training.log').open('a')
process=subprocess.Popen(cmd,cwd=REPO,stdout=log,stderr=subprocess.STDOUT,env=os.environ.copy())
print('PID',process.pid,'resume từ',done,'mục tiêu',TOTAL_STEPS)
# Sao lưu ở thread riêng: chỉ công bố checkpoint sau khi copy hoàn tất.
import threading
backup_stop=threading.Event()
def copy_atomic(src,dst):
    temporary=dst.with_name(dst.name+'.partial')
    shutil.copyfile(src,temporary)
    temporary.replace(dst)
def backup_loop():
    while True:
        try:
            for cp in RUN.glob('checkpoint_*_steps.zip'):
                dest=DRIVE_RUN/cp.name
                # Checkpoint is written before replay; wait for completed save.
                if time.time()-cp.stat().st_mtime<30:continue
                import zipfile
                if not dest.exists() and zipfile.is_zipfile(cp):copy_atomic(cp,dest)
            meta=RUN/'replay_buffer.json'
            if meta.exists():
                before=meta.read_bytes()
                step=json.loads(before)['step']
                checkpoint=DRIVE_RUN/f'checkpoint_{step}_steps.zip'
                remote=DRIVE_RUN/'replay_buffer.json'
                old=json.loads(remote.read_text()).get('step') if remote.exists() else -1
                if checkpoint.exists() and old!=step:
                    copy_atomic(RUN/'replay_buffer.pkl',DRIVE_RUN/'replay_buffer.pkl')
                    if meta.read_bytes()==before:copy_atomic(meta,remote)
                    else:remote.unlink(missing_ok=True)
            for name in ('manifest.json','run_config.json','progress.json'):
                src=RUN/name
                if src.exists():copy_atomic(src,DRIVE_RUN/name)
            if process.poll() is not None:
                final=RUN/'model.zip'
                if final.exists():copy_atomic(final,DRIVE_RUN/'model.zip')
                if not list(RUN.glob('checkpoint_*_steps.zip')) or all((DRIVE_RUN/p.name).exists() for p in RUN.glob('checkpoint_*_steps.zip')):break
        except Exception as error:
            (RUN/'backup_error.txt').write_text(str(error))
        time.sleep(15)
threading.Thread(target=backup_loop,daemon=True).start()
print('Sao lưu Drive chạy riêng; trước khi tắt runtime kiểm tra checkpoint và replay trên Drive.')


In [ ]:
# Bảng cố định: cập nhật tại chỗ, không clear_output toàn cell.
import time,json,html
from IPython.display import display,HTML
panel=display(HTML('<p>Đang đọc tiến độ...</p>'),display_id=True)
last_view=None
monitor_log=RUN/'training.log'
# Nếu cell launch cũ không ghi offset, chỉ đọc phiên OUTPUT gần nhất.
if 'session_log_offset' not in globals():
    tail=monitor_log.read_bytes() if monitor_log.exists() else b''
    marker=tail.rfind(b'OUTPUT:')
    session_log_offset=marker if marker>=0 else max(0,len(tail)-4000)
try:
    while True:
        progress_path=RUN/'progress.json'
        try: info=json.loads(progress_path.read_text()) if progress_path.exists() else {}
        except (OSError,ValueError): info={}
        status=process.poll() if 'process' in globals() else 'unknown'
        text=''
        if monitor_log.exists():
            with monitor_log.open('rb') as stream:
                stream.seek(min(session_log_offset,monitor_log.stat().st_size))
                text=stream.read().decode('utf-8',errors='replace')[-5000:]
        step=int(info.get('step',0));pct=min(100,100*step/TOTAL_STEPS)
        def fmt(value):return f'{value:.4f}' if isinstance(value,(int,float)) else '—'
        state='Đang chạy' if status is None else 'Không có tiến trình trong phiên notebook này' if status=='unknown' else 'Hoàn tất' if status==0 else 'Đã dừng an toàn' if status==3 else f'Đã dừng do lỗi · mã {status}'
        if status not in (None,0,3,'unknown') and 'KeyboardInterrupt' in text:
            state='Đã bị ngắt (KeyboardInterrupt)'
        J=info.get('mean_J');error=info.get('mean_param_error')
        reward=-J-.01-2*error if isinstance(J,(int,float)) and isinstance(error,(int,float)) else None
        view=(step,info.get('critic_loss'),status)
        if view!=last_view:
            panel.update(HTML(f"""<div style="border:1px solid #438b91;border-radius:12px;padding:16px">
<h3>EfficientNet V2-L epoch {EPOCH} + SAC</h3>
<progress value="{step}" max="{TOTAL_STEPS}" style="width:100%;height:24px"></progress>
<p><b>{step:,} / {TOTAL_STEPS:,} bước · {pct:.1f}% · {html.escape(state)}</b></p>
<p>J: {fmt(J)} · Reward: {fmt(reward)} · Sai số chuẩn hóa trong train: {fmt(error)}</p>
<p>Actor loss: {fmt(info.get('actor_loss'))} · Critic loss: {fmt(info.get('critic_loss'))}</p>
<p>Tốc độ: {fmt(info.get('env_steps_per_second'))} bước/giây · Còn khoảng {fmt(info.get('eta_hours')) if status is None else '—'} giờ</p>
<p>Log kỹ thuật được lưu trong training.log; không hiển thị trên bảng.</p>
</div>"""))
            last_view=view
        if status is not None:break
        time.sleep(3)
except KeyboardInterrupt:
    print('Dừng theo dõi; train vẫn chạy nền. Dùng STOP để dừng train an toàn.')


## Đánh giá checkpoint trên 16 mẫu train
Cùng seed khởi tạo; nhãn chỉ dùng báo cáo.

In [ ]:
# Chạy sau khi train kết thúc. Đánh giá trên chính 32 mẫu validation, không phải test độc lập.
assert 'process' not in globals() or process.poll() is not None, 'Chờ train kết thúc'
sys.path.insert(0,str(REPO))
from stable_baselines3 import SAC
from rlscm.backend import SCM
from rlscm.environment import PipeEnv
from rlscm.paths import LOW,HIGH
EVAL=RUN/'evaluation';EVAL.mkdir(exist_ok=True)
with h5py.File(VALID) as f: images=f['images'][:];truth=f['parameters'][:]
backend=SCM(EVAL,20,256,curve_points=256)
reports=[]
try:
    for cp in sorted(RUN.glob('checkpoint_*_steps.zip'),key=lambda p:int(p.stem.split('_')[1])):
        import gc
        if 'model' in globals():del model
        gc.collect();torch.cuda.empty_cache()
        model=SAC.load(cp,device='cuda',buffer_size=1)
        cases=[]
        for i,img in enumerate(images):
            env=PipeEnv(backend,[{'image':img.astype(np.int8)}],horizon=8,image_score=True,action_scale=.15,score_sigmas=(4,12))
            obs,_=env.reset(seed=20261113+i)
            initial=(LOW+env.x*(HIGH-LOW)).copy()
            for _ in range(8):obs,*_=env.step(model.predict(obs,deterministic=True)[0])
            cases.append(dict(case=i,initial=initial.tolist(),prediction=env.best_m.tolist(),truth=truth[i].tolist(),J=env.best_J))
        error=abs(np.array([c['prediction'] for c in cases])-truth)
        row=dict(checkpoint=cp.name,checkpoint_sha256=hashlib.sha256(cp.read_bytes()).hexdigest(),mean_normalized_error=float((error/(HIGH-LOW)).mean()),mare_percent=(100*error/abs(truth)).mean(0).tolist(),cases=cases)
        reports.append(row)
        (EVAL/'results.json').write_text(json.dumps(reports,indent=2))
        print(cp.name,row['mean_normalized_error'],row['mare_percent'])
finally:backend.close()


## Dừng an toàn
Chỉ chạy cell dưới khi muốn dừng. Chờ log STOPPED và tiến trình thoát trước khi tắt runtime.

In [ ]:
STOP.touch()
print('Đã yêu cầu lưu checkpoint + replay và dừng; kiểm tra log trước khi tắt runtime.')